In [2]:
import pickle
import numpy as np 
import pandas as pd 
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('TurkishSMSCollection.csv', on_bad_lines='skip', delimiter=';')

df['Group'] = np.where(df['Group']==1,1,0)

y = df['Group']

df.head()

,Message,Group,GroupText
0,125 lira,0,Normal
1,Baskanin aksam toplantısi fenaymis :),0,Normal
2,Bilal yalçnlara ne zaman gidiyoruz?,0,Normal
3,"BiP ile mesajlarimi aninda, daha eglenceli gon...",1,Spam
4,DIGITURKTEN FIRSAT! SiZE OZEL YIL SONUNA KADAR...,1,Spam


In [4]:
tokenizer=Tokenizer(num_words=500)
tokenizer.fit_on_texts(df['Message'])
sequences = tokenizer.texts_to_sequences(df['Message'])

pad = pad_sequences(sequences, maxlen=20)

In [5]:
x_train,x_test,y_train,y_test=train_test_split(
    pad,y,random_state=21,train_size=0.77
)

In [6]:
model = Sequential()

model.add(Embedding(500,100, input_length=20))
model.add(LSTM(64, dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

model.fit(
    x_train,y_train,
    validation_data=(x_test,y_test),
    epochs=8,
    batch_size=64
)

Epoch 1/8


c:\Users\Ahmet's Monster\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


58/58 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9415 - loss: 0.2363 - val_accuracy: 0.9799 - val_loss: 0.0604
Epoch 2/8
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9861 - loss: 0.0405 - val_accuracy: 0.9872 - val_loss: 0.0401
Epoch 3/8
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9915 - loss: 0.0267 - val_accuracy: 0.9918 - val_loss: 0.0310
Epoch 4/8
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9934 - loss: 0.0183 - val_accuracy: 0.9909 - val_loss: 0.0266
Epoch 5/8
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9962 - loss: 0.0122 - val_accuracy: 0.9899 - val_loss: 0.0241
Epoch 6/8
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9986 - loss: 0.0081 - val_accuracy: 0.9899 - val_loss: 0.0243
Epoch 7/8
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9978 - loss: 0.0063 - val_accuracy: 0.9890 - val_loss: 0.0306
Epoch 8/8
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9992 - loss: 0.0045 - val_accuracy: 0.9899 - val_loss: 0.0261


In [7]:
message = ['SOK SOK SOK BIR ALANA BIR BEDAVA']

message_tokenizer = tokenizer.texts_to_sequences(message)
message_tokenizer = pad_sequences(message_tokenizer,maxlen=20)

model.predict(message_tokenizer)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step


array([[0.79245365]], dtype=float32)

In [ ]:
model.save('backend/spam_model.h5')

In [ ]:
with open('backend/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)